# UC-01 · Select a pump for given conditions

**Use case:** [UC-01](../docs/product/use-cases.md) - *given a duty (Q, H) and a
catalogue of known pump curves, rank the pumps that can serve it, as they are, with
a speed change (VFD) or with an impeller trim.*

The catalogue is a folder of `*.pumpcurve.json` files (value + unit pairs).  This
notebook builds three small curves, writes them to a temporary folder, loads them
with `Catalogue.from_folder` and calls `Catalogue.select`.  Ranking: pumps inside the
preferred operating region (POR) first, then higher efficiency at the duty, then the
smaller distance from the best-efficiency point.

> The POR (70-120 % of the BEP flow) comes from a secondary source and is flagged
> `verified = false`.  Viscosity correction is **not** applied by the selection.

**Acceptance:** every candidate passes through the duty head at the duty flow (1 %
for an as-is curve, exactly for speed / trim); the ranking order is respected; a
pump that cannot reach the duty is rejected with a reason.

## 1 · Setup

In [ ]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from pump import Q_, Fluid, Water, TestPoint, PerformanceCurve

import tempfile
from pathlib import Path

from pump import AcceptanceCriteria, Catalogue, CatalogueEntry
from pump.catalogue import load_entry, save_entry

## 2 · Three catalogue curves

Each entry carries the curve, the speed and impeller diameter it was measured at, and
the allowed diameter range.  `P-1002A` is a larger pump (flow x 1.3, head x 0.85) and
`P-0500S` a small one that cannot reach the duty.

In [ ]:
# Illustrative water test of pump P-1001A (1480 rpm):
#          Q [m3/h]  Head [m]  P [kW]  eta [%]
P1001A = [
    (160.0, 125.37, 77.97, 70.03),
    (240.0, 119.11, 95.17, 81.82),
    (320.0, 110.79, 114.44, 84.44),
    (400.0, 100.01, 147.26, 74.00),
    (480.0, 86.72, 173.98, 65.13),
    (540.0, 75.60, 194.76, 57.14),
]
water = Fluid("Test water", density=Q_(1000.0, "kg/m**3"))


def build_curve(rows, fluid, speed, degree=3, head_k=1.0, power_k=1.0, eff_k=1.0):
    "Measured rows -> PerformanceCurve with Head / eta pinned on the points."
    points = [
        TestPoint(
            fluid=fluid,
            capacity=Q_(q, "m**3/h"),
            speed_of_rotation=speed,
            _head=Q_(h * head_k, "m"),
            _efficiency=Q_(e * eff_k, "percent"),
            breaking_power=Q_(p * power_k, "kW"),
        )
        for q, h, p, e in rows
    ]
    return PerformanceCurve(fluid, points, polynomial_degree=degree)


def make_entry(name, speed, diameter, d_min, d_max, q_scale=1.0, h_scale=1.0):
    rows = [(q * q_scale, h * h_scale, p * q_scale * h_scale, e) for q, h, p, e in P1001A]
    return CatalogueEntry(
        name=name,
        curve=build_curve(rows, water, speed),
        speed=speed,
        diameter=diameter,
        d_min=d_min,
        d_max=d_max,
    )


rpm = Q_(1480, "rpm")
entries = [
    make_entry("P-1001A", rpm, Q_(300, "mm"), Q_(270, "mm"), Q_(310, "mm")),
    make_entry("P-1002A", rpm, Q_(340, "mm"), Q_(300, "mm"), Q_(350, "mm"), 1.3, 0.85),
    make_entry("P-0500S", rpm, Q_(180, "mm"), Q_(160, "mm"), Q_(190, "mm"), 0.2, 1.0),
]
for e in entries:
    print(e.name, e.speed, e.diameter, "has efficiency:", e.has_efficiency)

## 3 · Write and load the catalogue

`save_entry` writes one `*.pumpcurve.json` per pump; `Catalogue.from_folder` reads the
folder back.  A malformed file never aborts the load: it is listed in `load_errors`.

In [ ]:
tmp = tempfile.TemporaryDirectory()
catalogue_dir = Path(tmp.name)
for e in entries:
    save_entry(e, catalogue_dir)

catalogue = Catalogue.from_folder(catalogue_dir)
print("files    :", sorted(p.name for p in catalogue_dir.glob("*.pumpcurve.json")))
print("entries  :", catalogue.names)
print("errors   :", catalogue.load_errors)

reloaded = load_entry(catalogue_dir / "P-1001A.pumpcurve.json")
print("round trip speed:", reloaded.speed, "| diameter:", reloaded.diameter)

## 4 · Select for a duty

Duty: 300 m3/h at 105 m of differential head, the user's engineering units
(here 1321 gpm and 344 ft would give the same result).

In [ ]:
duty_q = Q_(300.0, "m**3/h")
duty_h = Q_(105.0, "m")

ranked = catalogue.select(duty_q, duty_h)

print(f"{'#':>2} {'candidate':<26} {'how':<7} {'eta %':>6} {'P kW':>7} {'Q/Qbep':>7} {'POR':>4}")
for i, c in enumerate(ranked, start=1):
    power = c.power_at_duty.magnitude if c.power_at_duty is not None else float("nan")
    print(
        f"{i:>2} {c.label:<26} {c.adaptation:<7} {c.eta_at_duty.magnitude:6.1f} "
        f"{power:7.1f} {c.q_over_qbep:7.2f} {str(c.in_por):>4}"
    )

print("\nRejected:")
for reason in ranked.reasons:
    print(" -", reason[:140])

## 5 · Plot the best candidates

The adapted head curves of the three best candidates all pass through the duty point.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for c in ranked[:3]:
    caps = np.asarray(c.curve.fitter.capacities, dtype=float)
    xs = np.linspace(caps.min(), caps.max(), 200)
    ax.plot(xs, np.polyval(c.curve.fitter.head_coeffs, xs), label=c.label)
ax.plot(duty_q.magnitude, duty_h.magnitude, "k*", ms=14, label="duty")
ax.set_xlabel("Capacity (m3/h)")
ax.set_ylabel("Head (m)")
ax.set_title("UC-01 - candidates adapted to the duty")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## 6 · The preferred operating region

The POR bounds are data (`AcceptanceCriteria.selection`), not code.

In [ ]:
crit = AcceptanceCriteria.default()
sel = crit.selection
print(f"POR = [{sel.por_min:g}, {sel.por_max:g}] x Q_BEP  | verified = {sel.verified}")
print("source:", sel.source[:110], "...")

## 7 · Acceptance

Final assertions: the demo becomes a test.

In [ ]:
# the catalogue loaded all three files
assert catalogue.names == ["P-0500S", "P-1001A", "P-1002A"]
assert catalogue.load_errors == []
assert reloaded.speed.to("rpm").magnitude == 1480

# at least two pumps can serve the duty
assert len(ranked) >= 2

# every candidate passes through the duty head at the duty flow
for c in ranked:
    h = c.curve.predict_head(duty_q).to("m").magnitude
    assert abs(h - duty_h.magnitude) / duty_h.magnitude < 0.01, (c.label, h)
    assert c.adaptation in ("as-is", "speed", "trim")
    if c.adaptation == "speed":
        assert 0.9 * 1480 < c.value.to("rpm").magnitude < 1.1 * 1480
    if c.adaptation == "trim":
        assert c.entry.d_min <= c.value <= c.entry.d_max

# ranking: in-POR candidates first, then non-increasing efficiency
for a, b in zip(ranked, ranked[1:]):
    assert a.in_por or not b.in_por
    if a.in_por == b.in_por:
        assert a.eta_at_duty.magnitude >= b.eta_at_duty.magnitude - 1e-3

# the small pump cannot reach the duty and says why
assert "P-0500S" in {r.name for r in ranked.rejected}
assert all("P-0500S" != c.name for c in ranked)

# the catalogue entries are not mutated by the selection
assert catalogue.entries[1].speed.to("rpm").magnitude == 1480
assert sel.verified is True  # API 610 12th ed.: POR 70-120 % of BEP, checked

tmp.cleanup()
print("UC-01 acceptance OK:", [c.label for c in ranked[:3]])